# Análisis Exploratorio de Datos (EDA) — Resultados de Análisis de Laboratorio de Suelos en Colombia (AGROSAVIA)
## Etapa 1: Justificación, Diagnóstico Exhaustivo y Limpieza de Datos

### 1. Justificación y ficha técnica del conjunto de datos

* **Entidad productora y fuente:** Corporación Colombiana de Investigación Agropecuaria (**AGROSAVIA**), publicado en el portal oficial **Datos Abiertos Colombia** (`datos.gov.co`).
* **URL oficial:** <https://www.datos.gov.co/Agricultura-y-Desarrollo-Rural/Resultados-de-An-lisis-de-Laboratorio-Suelos-en-Co/ch4u-f3i5/about_data>
* **Licencia declarada:** `Creative Commons Attribution | Share Alike 4.0 International` (CC BY-SA 4.0).
* **Archivo local utilizado:** `data_raw/Resultados_de_Análisis_de_Laboratorio_Suelos_en_Colombia_20261123.csv` (26.39 MB).
* **Dimensiones iniciales:** **92,738 filas × 32 columnas**, con cobertura nacional en los **32 departamentos de Colombia y Bogotá, D.C.**, recopilando análisis de laboratorio registrados entre **enero de 2014 y septiembre de 2025**. El conjunto supera ampliamente los criterios mínimos de la rúbrica (> 200 filas y > 8 columnas), combina variables temporales, geográficas, agronómicas y 21 determinaciones fisicoquímicas, y presenta imperfecciones reales propias de registros de laboratorio.

#### Pertinencia en ciencias biológicas y edafología
El suelo constituye la interfaz biogeoquímica fundamental entre la litosfera, la atmósfera, la hidrosfera y la biosfera. En sistemas agroecológicos tropicales como los de Colombia, parámetros como el **pH agua:suelo**, la **acidez y el aluminio intercambiable** (determinantes de toxicidad alumínica y disponibilidad de fósforo en suelos ácidos), la **materia orgánica** (reservorio principal de carbono edáfico y sustrato energético para la microbiota del suelo), la **capacidad de intercambio catiónico (CIC)**, la **conductividad eléctrica** (indicadora de salinidad/sodicidad) y la disponibilidad de macronutrientes ($\text{P}, \text{S}, \text{Ca}, \text{Mg}, \text{K}$) y micronutrientes ($\text{Fe}, \text{Cu}, \text{Mn}, \text{Zn}, \text{B}$) permiten caracterizar la fertilidad potencial, las limitantes fisiológicas para distintos cultivos y la heterogeneidad espacial y temporal de los agroecosistemas colombianos.

#### Herramientas y declaración de transparencia
* **Librerías de análisis:** Todo el diagnóstico, limpieza y transformación se ejecuta exclusivamente con `pandas` (con soporte de `numpy`, `re` y `unicodedata` de Python para expresiones regulares y normalización Unicode) sobre un entorno virtual reproducible (`.venv`, Python 3.14). En etapas posteriores se utilizarán `matplotlib` y `seaborn` para las visualizaciones obligatorias.
* **Declaración de uso de IA:** Conforme a las instrucciones de la prueba, se declara el uso de un asistente de inteligencia artificial como apoyo para organizar las secciones del cuaderno, redactar y depurar código en `pandas` y pulir la redacción. Todas las decisiones edafológicas, reglas de limpieza aplicadas e interpretaciones fueron definidas y verificadas directamente por el autor sobre los datos reales del archivo CSV.

---
## 2. Carga inicial y diagnóstico exhaustivo con `pandas`

### 2.1 Carga, codificación y auditoría de encabezados
Antes de leer el archivo con `pandas`, se verifica el tamaño en disco, la presencia o ausencia de BOM (*Byte Order Mark*) UTF-8 y los primeros registros en bruto. Se carga explícitamente **sin** el parámetro `thousands=','` porque, como se demostrará en la sección 2.2, la coma `,` cumple dos roles opuestos dentro del mismo archivo (separador de miles en `Secuencial` y separador decimal en micronutrientes por doble ácido).

In [1]:
import os
import re
import unicodedata
import difflib
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 140)

RAW_PATH = 'data_raw/Resultados_de_Análisis_de_Laboratorio_Suelos_en_Colombia_20261123.csv'
PROCESSED_PATH = 'data_processed/suelos_limpios.csv'

file_bytes = os.path.getsize(RAW_PATH)
with open(RAW_PATH, 'rb') as f:
    raw_head = f.read(600)

print(f'Tamaño del archivo: {file_bytes:,} bytes ({file_bytes / (1024**2):.2f} MB)')
print('Primeros 6 bytes (verificación de BOM UTF-8):', raw_head[:6])

df_raw = pd.read_csv(RAW_PATH, low_memory=False)
print(f'Dimensiones iniciales (df_raw): {df_raw.shape[0]:,} filas x {df_raw.shape[1]} columnas')
display(df_raw.head(5))

Tamaño del archivo: 26,390,661 bytes (25.17 MB)
Primeros 6 bytes (verificación de BOM UTF-8): b'"Secue'


Dimensiones iniciales (df_raw): 92,738 filas x 32 columnas


,Secuencial,Fecha de Análisis,Departamento,Municipio,Cultivo,Estado,Tiempo de establecimiento,Topografia,Drenaje,Riego,Fertilizantes aplicados,pH agua:suelo,Materia organica,Fósforo Bray II,Azufre Fosfato monocalcico,Acidez Intercambiable,Aluminio intercambiable,Calcio intercambiable,Magnesio intercambiable,Potasio intercambiable,Sodio intercambiable,capacidad de intercambio cationico,Conductividad electrica,Hierro disponible olsen,Cobre disponible,Manganeso disponible Olsen,Zinc disponible Olsen,Boro disponible,Hierro disponible doble acido,Cobre disponible doble acido,Manganeso disponible doble acido,Zinc disponible doble acido
0,1,7/01/2014,NARIÑO,SAN ANDRÉS DE TUMACO,No Indica,No indica,No indica,No indica,No indica,No indica,No indica,5.64,3.889,6.482,8.384,ND,ND,1.541,0.380,0.118,0.054,2.094,0.133,66.39,1.700,1.700,1.5,0.187,ND,ND,ND,ND
1,2,21/08/2014,HUILA,SANTA MARÍA,Granadilla,Establecido,De 1 a 5 años,Pendiente,Buen drenaje,No Tiene,No indica,4.24,2.287,520.1,90.30,2.021,1.483,2.932,0.925,2.131,0.043,8.054,2.749,250.9,3.7,29.4,28.70,2.084,ND,ND,ND,ND
2,3,22/08/2014,ANTIOQUIA,LIBORINA,Café,Por establecer,No indica,Pendiente,Mal drenaje,No Tiene,PRODUCCION,5.22,2.518,58.11,2.686,0.955,0.796,3.818,1.110,<0.09,0.010,5.985,0.328,390,11.10,8.299,5,0.085,ND,ND,ND,ND
3,4,22/08/2014,ANTIOQUIA,LIBORINA,Maracuyá,Por establecer,No indica,Ondulado,Mal drenaje,No Tiene,No indica,5.45,2.841,4.583,1.782,0,0,4.754,3.577,0.118,0.010,8.461,0.171,200,4.2,4.699,1,0.229,ND,ND,ND,ND
4,5,22/08/2014,ANTIOQUIA,LIBORINA,Café,Por establecer,No indica,Ondulado,Mal drenaje,No Tiene,PRODUCCION,5.43,2.703,5.512,3.138,0.150,0.109,7.338,3.022,0.096,0.032,10.64,0.323,117.9,3.8,4.600,0.700,0.139,ND,ND,ND,ND


In [2]:
# Auditoría exacta de nombres de columnas (espacios no rompibles, mayúsculas, tildes y símbolos)
header_audit = []
for idx, col in enumerate(df_raw.columns):
    has_nbsp = '\xa0' in col
    has_colon = ':' in col
    has_accent = any(ord(ch) > 127 for ch in col)
    header_audit.append({
        'indice': idx,
        'nombre_crudo': col,
        'repr_exacto': repr(col),
        'contiene_nbsp_xa0': has_nbsp,
        'contiene_dos_puntos': has_colon,
        'caracteres_no_ascii': has_accent,
    })

df_headers = pd.DataFrame(header_audit)
print('Columnas con caracteres especiales ocultos (\xa0) o puntuación irregular:')
display(df_headers[df_headers['contiene_nbsp_xa0'] | df_headers['contiene_dos_puntos'] | (df_headers['nombre_crudo'].str[0].str.islower())])

Columnas con caracteres especiales ocultos ( ) o puntuación irregular:


,indice,nombre_crudo,repr_exacto,contiene_nbsp_xa0,contiene_dos_puntos,caracteres_no_ascii
11,11,pH agua:suelo,'pH agua:suelo',False,True,False
21,21,capacidad de intercambio cationico,'capacidad de intercambio cationico',False,False,False
31,31,Zinc disponible doble acido,'Zinc disponible doble \xa0acido',True,False,True


**Observación 2.1:** El archivo está codificado en UTF-8 sin BOM y carga limpiamente (`92,738 × 32`). Sin embargo, los encabezados presentan irregularidades reales:
1. `'Zinc disponible doble \xa0acido'` contiene un **espacio no rompible (`\xa0`, U+00A0)** antes de `acido`, invisible al imprimir pero capaz de provocar errores `KeyError` si no se normaliza.
2. Inconsistencia de mayúsculas/minúsculas (`'capacidad de intercambio cationico'` en minúscula inicial; `'Hierro disponible olsen'` frente a `'Manganeso disponible Olsen'`).
3. Uso mixto de tildes (`'Fósforo'`, `'Análisis'` frente a `'organica'`, `'monocalcico'`, `'acido'`) y dos puntos (`'pH agua:suelo'`).

---
### 2.2 Diagnóstico de tipos mal inferidos y causas de falla de parseo

In [3]:
# Resumen de tipos inferidos por pandas
dtype_summary = pd.DataFrame({
    'columna': df_raw.columns,
    'dtype_inferido': df_raw.dtypes.astype(str).values,
    'nulos_explicitos': df_raw.isna().sum().values,
    'valores_unicos': [df_raw[c].nunique(dropna=True) for c in df_raw.columns]
})
print('Conteo por tipo inferido:', dtype_summary['dtype_inferido'].value_counts().to_dict())
display(dtype_summary)

Conteo por tipo inferido:

 {'str': 32}


,columna,dtype_inferido,nulos_explicitos,valores_unicos
0,Secuencial,str,0,92738
1,Fecha de Análisis,str,0,2388
2,Departamento,str,0,33
3,Municipio,str,2,990
4,Cultivo,str,0,282
5,Estado,str,0,4
6,Tiempo de establecimiento,str,0,9
7,Topografia,str,0,13
8,Drenaje,str,0,6
9,Riego,str,0,17


In [4]:
# Diagnóstico de Secuencial y Fecha de Análisis
non_pure_digits = ~df_raw['Secuencial'].astype(str).str.match(r'^\d+$', na=False)
print(f'Secuencial con formato de miles (coma): {non_pure_digits.sum():,} de {len(df_raw):,} filas')
print('Muestras de Secuencial:', df_raw.loc[non_pure_digits, 'Secuencial'].head(6).tolist())

s_fecha = df_raw['Fecha de Análisis'].astype(str).str.strip()
patrones_fecha = s_fecha.str.replace(r'\d', 'D', regex=True).value_counts()
fecha_dmy = pd.to_datetime(s_fecha, format='%d/%m/%Y', errors='coerce')
fecha_default = pd.to_datetime(s_fecha, errors='coerce')

print('\nPatrones de Fecha de Análisis:')
print(patrones_fecha.to_dict())
print(f'NaT con format="%d/%m/%Y": {fecha_dmy.isna().sum()} | Rango: {fecha_dmy.min().date()} a {fecha_dmy.max().date()}')
print(f'NaT si se omitiera format/dayfirst (error potencial): {fecha_default.isna().sum()}')

Secuencial con formato de miles (coma): 91,739 de 92,738 filas
Muestras de Secuencial: ['1,000', '1,001', '1,002', '1,003', '1,004', '1,005']

Patrones de Fecha de Análisis:
{'DD/DD/DDDD': 67145, 'D/DD/DDDD': 25593}
NaT con format="%d/%m/%Y": 0 | Rango: 2014-01-07 a 2025-09-05
NaT si se omitiera format/dayfirst (error potencial): 57627


In [5]:
# Diagnóstico detallado de las 21 columnas fisicoquímicas
chem_cols_raw = list(df_raw.columns[11:])

num_re = re.compile(r'^-?\d+(\.\d+)?$')
lt_re = re.compile(r'^<\s*\d+([.,]\d+)?$')
gt_re = re.compile(r'^>\s*\d+([.,]\d+)?$')
comma_re = re.compile(r'^-?\d+,\d+$')
trail_dot_re = re.compile(r'^-?\d+\.$')

chem_diag = []
for col in chem_cols_raw:
    s = df_raw[col]
    n_na = int(s.isna().sum())
    s_str = s.dropna().astype(str).str.strip()
    is_nd = s_str.str.upper().eq('ND')
    is_num = s_str.str.match(num_re)
    is_lt = s_str.str.match(lt_re)
    is_gt = s_str.str.match(gt_re)
    is_comma = s_str.str.match(comma_re)
    is_trail = s_str.str.match(trail_dot_re)
    other = s_str[~(is_nd | is_num | is_lt | is_gt | is_comma | is_trail)]

    chem_diag.append({
        'columna': col,
        'NaN': n_na,
        'ND_conteo': int(is_nd.sum()),
        'ND_pct': round(is_nd.sum() / len(df_raw) * 100, 2),
        'censurado_LT': int(is_lt.sum()),
        'censurado_GT': int(is_gt.sum()),
        'coma_decimal': int(is_comma.sum()),
        'punto_final': int(is_trail.sum()),
        'tokens_raros': len(other),
        'ejemplos_LT_GT': dict(s_str[is_lt | is_gt].value_counts().head(3)),
        'ejemplos_raros': dict(other.value_counts().head(5)),
    })

df_chem_diag = pd.DataFrame(chem_diag)
display(df_chem_diag)

,columna,NaN,ND_conteo,ND_pct,censurado_LT,censurado_GT,coma_decimal,punto_final,tokens_raros,ejemplos_LT_GT,ejemplos_raros
0,pH agua:suelo,0,0,0.00,6,4,0,0,1,"{'<3.80': 6, '>10.00': 4}",{'4..01': 1}
1,Materia organica,0,2,0.00,69,2,0,0,0,"{'<0.38': 69, '>44.82': 2}",{}
2,Fósforo Bray II,0,1469,1.58,13564,0,0,53,0,"{'<3.87': 11366, '<1.29': 2097, '<1.42': 101}",{}
3,Azufre Fosfato monocalcico,0,1499,1.62,63,0,0,20,0,"{'<0.01': 61, '<2.50': 2}",{}
4,Acidez Intercambiable,0,45130,48.66,0,0,0,0,0,{},{}
5,Aluminio intercambiable,0,45130,48.66,0,0,0,0,0,{},{}
6,Calcio intercambiable,0,0,0.00,5170,0,0,0,0,"{'<1.12': 2340, '< 0.5': 1592, '<0.59': 1232}",{}
7,Magnesio intercambiable,0,0,0.00,3875,0,0,0,0,{'<0.20': 3875},{}
8,Potasio intercambiable,0,0,0.00,15167,0,0,0,0,{'<0.09': 15167},{}
9,Sodio intercambiable,0,0,0.00,44774,0,0,0,0,"{'<0.14': 33005, '< 0.1': 11763, '< 0.0': 6}",{}


In [6]:
# Evidencia empírica del doble significado de la coma (,) en Hierro y Zinc doble ácido
mask_fe_comma = df_raw['Hierro disponible doble acido'].fillna('').str.contains(',')
print(f'Registros con coma en "Hierro disponible doble acido": {mask_fe_comma.sum()}')
display(df_raw.loc[mask_fe_comma, [
    'Secuencial', 'Fecha de Análisis', 'Departamento',
    'Hierro disponible doble acido', 'Cobre disponible doble acido',
    'Manganeso disponible doble acido', 'Zinc disponible doble \xa0acido'
]].head(10))

Registros con coma en "Hierro disponible doble acido": 42


,Secuencial,Fecha de Análisis,Departamento,Hierro disponible doble acido,Cobre disponible doble acido,Manganeso disponible doble acido,Zinc disponible doble acido
711,712,26/09/2014,META,"8,039",0.96,0.2,"0,04"
765,766,26/09/2014,META,"9,800",0.8,0.32,"0,04"
1722,"1,723",13/06/2014,META,"4,519",0.560,7,"7,24"
1725,"1,726",13/06/2014,META,"9,039",1.48,1.24,"8,4"
1729,"1,730",13/06/2014,META,"9,199",1.32,1.08,"11,88"
1930,"1,931",13/06/2014,META,"8,039",1.36,2.48,"6,36"
1932,"1,933",13/06/2014,META,"9,039",1.32,2.6,"16,8"
1934,"1,935",13/06/2014,META,"8,039",1,3.04,"0,44"
1943,"1,944",13/06/2014,META,"9,880",0.88,0.96,"2,84"
2701,"2,702",27/11/2014,VICHADA,"8,279",0.560,0.16,"0,24"


**Hallazgos críticos 2.2:**
* **32 de 32 columnas** fueron inferidas como texto (`str`).
* **`Secuencial`** tiene coma como separador de miles en `91,739` filas (`1,000` a `92,738`).
* **`Fecha de Análisis`** sigue estrictamente el formato `D/MM/YYYY` o `DD/MM/YYYY` (`%d/%m/%Y`, `0` valores `NaT`). Usar `pd.to_datetime` sin especificar `format='%d/%m/%Y'` (o `dayfirst=True`) falla o invierte día/mes cuando el día supera 12.
* **Coma decimal vs. separador de miles:** `Zinc disponible doble \xa0acido` usa coma decimal en **4,732** registros (incluyendo `< 0,4`, `0,24`, `1,120`) y `Hierro disponible doble acido` tiene **42** registros con coma decimal (`2,078` a `9,918`). Al inspeccionar esos mismos lotes junto a `Cobre` y `Manganeso` (que sí usan punto decimal, e.g., `0.96`, `0.2`), se comprueba que corresponden a valores menores que `10 mg/kg` (no a miles): si se hubiera leído el CSV con `thousands=','`, esos valores se habrían multiplicado erróneamente por `1,000`.
* **Censura analítica (`<LD` y `>UL`):** aparecen límites de detección inferiores en 15 analitos (destacando `Sodio intercambiable` con `44,774` `<LD`, `Zinc disponible Olsen` con `32,740`, `Cobre disponible` con `23,412`, `Potasio intercambiable` con `15,167` y `Fósforo Bray II` con `13,564`) y cotas superiores `>UL` en `pH agua:suelo` (`>10.00`, 4 filas), `Materia organica` (`>44.82`, 2 filas) y `Conductividad electrica` (`>12.88`, 86 filas).
* **Tokens anómalos puntuales:** punto decimal duplicado `'4..01'` en `pH agua:suelo` (1 fila), letra `'b'` en `Boro disponible` (1 fila), y enteros de 4 dígitos terminados en punto (`'1004.'`, `'1236.'`, etc., que son válidos al convertirse con `float`).

---
### 2.3 Faltantes explícitos, centinelas implícitos y faltantes estructurales de protocolo

In [7]:
# Tabla consolidada de faltantes explícitos (NaN) vs. centinelas implícitos ('ND', 'No indica', 'No Indica')
missing_rows = []
for col in df_raw.columns:
    s = df_raw[col]
    n_nan = int(s.isna().sum())
    s_clean = s.dropna().astype(str).str.strip()
    n_nd = int(s_clean.str.upper().eq('ND').sum())
    n_no_ind = int(s_clean.str.lower().eq('no indica').sum())
    n_total_miss = n_nan + n_nd + n_no_ind
    missing_rows.append({
        'columna': col,
        'NaN_explicitos': n_nan,
        'ND_laboratorio': n_nd,
        'No_indica_texto': n_no_ind,
        'faltantes_reales_total': n_total_miss,
        'faltantes_reales_pct': round(n_total_miss / len(df_raw) * 100, 2),
    })

df_missing = pd.DataFrame(missing_rows).sort_values('faltantes_reales_pct', ascending=False)
print(f'Total de celdas NaN explícitas en todo el dataset crudo: {int(df_raw.isna().sum().sum())}')
display(df_missing[df_missing['faltantes_reales_total'] > 0])

Total de celdas NaN explícitas en todo el dataset crudo: 3


,columna,NaN_explicitos,ND_laboratorio,No_indica_texto,faltantes_reales_total,faltantes_reales_pct
31,Zinc disponible doble acido,0,87952,0,87952,94.84
30,Manganeso disponible doble acido,0,87952,0,87952,94.84
29,Cobre disponible doble acido,0,87952,0,87952,94.84
28,Hierro disponible doble acido,1,87951,0,87952,94.84
10,Fertilizantes aplicados,0,0,50200,50200,54.13
15,Acidez Intercambiable,0,45130,0,45130,48.66
16,Aluminio intercambiable,0,45130,0,45130,48.66
9,Riego,0,0,39168,39168,42.24
6,Tiempo de establecimiento,0,0,30814,30814,33.23
8,Drenaje,0,0,25605,25605,27.61


In [8]:
# Verificación de dos hipótesis de protocolo de laboratorio mediante tablas cruzadas
ph_tmp = pd.to_numeric(
    df_raw['pH agua:suelo'].astype(str).str.strip().str.replace('4..01', '4.01', regex=False).str.replace(r'^[<>]\s*', '', regex=True),
    errors='coerce'
)
ac_nd = df_raw['Acidez Intercambiable'].astype(str).str.strip().str.upper().eq('ND')
al_nd = df_raw['Aluminio intercambiable'].astype(str).str.strip().str.upper().eq('ND')

print('1) ¿Acidez ND coincide exactamente con Aluminio ND?:', (ac_nd == al_nd).all())
print('\nTabla cruzada: pH <= 5.5 vs. Acidez/Aluminio == ND:')
ct_acidez = pd.crosstab(
    ph_tmp <= 5.5, ac_nd,
    rownames=['pH <= 5.5'], colnames=['Acidez_Aluminio_ND'], margins=True
)
display(ct_acidez)

fe_olsen_nd = df_raw['Hierro disponible olsen'].astype(str).str.strip().str.upper().eq('ND')
fe_da_nd = df_raw['Hierro disponible doble acido'].fillna('ND').astype(str).str.strip().str.upper().eq('ND')
print('\n2) Tabla cruzada: Hierro Olsen ND vs. Hierro Doble Ácido ND (métodos mutuamente excluyentes):')
ct_metodos = pd.crosstab(
    fe_olsen_nd, fe_da_nd,
    rownames=['Olsen_ND'], colnames=['Doble_Acido_ND'], margins=True
)
display(ct_metodos)

1) ¿Acidez ND coincide exactamente con Aluminio ND?: True

Tabla cruzada: pH <= 5.5 vs. Acidez/Aluminio == ND:


Acidez_Aluminio_ND,False,True,All
pH <= 5.5,,,
False,176,44730,44906
True,47432,400,47832
All,47608,45130,92738



2) Tabla cruzada: Hierro Olsen ND vs. Hierro Doble Ácido ND (métodos mutuamente excluyentes):


Doble_Acido_ND,False,True,All
Olsen_ND,,,
False,0,86866,86866
True,4786,1086,5872
All,4786,87952,92738


**Hallazgo estructural 2.3:** Una revisión ingenua con `.isna().sum()` reportaría apenas **3 celdas faltantes** (`2` en `Municipio` y `1` en `Hierro disponible doble acido`), ocultando cientos de miles de faltantes codificados como `'ND'` o `'No indica'`. Más importante aún, las tablas cruzadas prueban que la mayoría de `'ND'` fisicoquímicos **no son aleatorios**, sino resultado de **dos protocolos de laboratorio**:
1. **Acidez y Aluminio intercambiables (`45,130` `'ND'`, `48.66%`):** De los `45,130` registros `'ND'`, **`44,730` (`99.1%`) corresponden a suelos con `pH > 5.5`**, donde el aluminio intercambiable precipita y el protocolo estándar no ejecuta titulación de acidez/Al.
2. **Olsen vs. Doble ácido:** **0 muestras** tienen ambos métodos simultáneamente (`86,866` evaluadas por Olsen, `4,786` por Doble ácido y `1,086` sin ninguno). Imputar estas columnas por la media global destruiría el significado químico del protocolo; deben conservarse como `NaN` y acompañarse de una variable indicadora del método.

---
### 2.4 Diagnóstico de filas duplicadas

In [9]:
# Verificación de duplicados exactos vs. duplicados ignorando el identificador Secuencial
dup_exact = int(df_raw.duplicated().sum())
cols_sin_seq = [c for c in df_raw.columns if c != 'Secuencial']
dup_sin_seq_mask = df_raw.duplicated(subset=cols_sin_seq, keep='first')
dup_grupos_mask = df_raw.duplicated(subset=cols_sin_seq, keep=False)

print(f'Duplicados exactos (incluyendo Secuencial): {dup_exact}')
print(f'Filas duplicadas excluyendo Secuencial: {int(dup_sin_seq_mask.sum()):,}')
print(f'Total de filas involucradas en grupos duplicados: {int(dup_grupos_mask.sum()):,}')

grp_sizes = df_raw.groupby(cols_sin_seq, dropna=False).size()
print('Tamaño de los grupos duplicados:', grp_sizes[grp_sizes > 1].value_counts().to_dict())

print('\nMuestra de registros duplicados (idénticos en fecha, ubicación, cultivo y 21 variables químicas):')
display(df_raw.loc[dup_grupos_mask, ['Secuencial', 'Fecha de Análisis', 'Departamento', 'Municipio', 'Cultivo', 'pH agua:suelo', 'Materia organica', 'Fósforo Bray II']].sort_values(['Fecha de Análisis', 'Departamento', 'Municipio', 'pH agua:suelo']).head(10))

dup_years = pd.to_datetime(df_raw.loc[dup_sin_seq_mask, 'Fecha de Análisis'], format='%d/%m/%Y').dt.year.value_counts().sort_index()
print('\nDistribución por año de las 2,046 filas duplicadas redundantes:')
display(dup_years.to_frame('filas_duplicadas'))

Duplicados exactos (incluyendo Secuencial): 0
Filas duplicadas excluyendo Secuencial: 2,046
Total de filas involucradas en grupos duplicados: 4,068


Tamaño de los grupos duplicados: {2: 2010, 4: 12}

Muestra de registros duplicados (idénticos en fecha, ubicación, cultivo y 21 variables químicas):


,Secuencial,Fecha de Análisis,Departamento,Municipio,Cultivo,pH agua:suelo,Materia organica,Fósforo Bray II
6674,"6,675",1/04/2014,META,ACACÍAS,Pastos,4.63,2.035,2.657
6683,"6,684",1/04/2014,META,ACACÍAS,Pastos,4.63,2.035,2.657
6794,"6,795",1/04/2014,META,VILLAVICENCIO,Pastos,5.93,2.627,3.480
6795,"6,796",1/04/2014,META,VILLAVICENCIO,Pastos,5.93,2.627,3.480
42536,"42,527",1/06/2017,CAUCA,PATÍA,Pastos,5.99,3.161,<3.87
42537,"42,528",1/06/2017,CAUCA,PATÍA,Pastos,5.99,3.161,<3.87
42534,"42,525",1/06/2017,CAUCA,SILVIA,Pastos,4.92,18.54,3.513
42535,"42,526",1/06/2017,CAUCA,SILVIA,Pastos,4.92,18.54,3.513
42527,"42,518",1/06/2017,CAUCA,SOTARÁ,Pastos,5.48,13.56,5.779
42528,"42,519",1/06/2017,CAUCA,SOTARÁ,Pastos,5.48,13.56,5.779



Distribución por año de las 2,046 filas duplicadas redundantes:


,filas_duplicadas
Fecha de Análisis,
2014,152
2015,344
2016,384
2017,571
2018,592
2019,3


**Hallazgo 2.4:** `df_raw.duplicated().sum()` devuelve `0` únicamente porque el consecutivo `Secuencial` asigna un número distinto a cada fila. Al excluir `Secuencial`, emergen **2,046 filas redundantes** (`4,068` registros agrupados en `2,010` pares idénticos y `12` cuádruples), concentradas en el periodo **2014–2019**.

---
### 2.5 Diagnóstico de categorías escritas de varias formas

In [10]:
def norm_key(s):
    if pd.isna(s):
        return None
    s = re.sub(r'\s+', ' ', str(s).strip())
    s = ''.join(c for c in unicodedata.normalize('NFKD', s) if not unicodedata.combining(c))
    return s.lower()

cat_cols_raw = [
    'Departamento', 'Municipio', 'Cultivo', 'Estado',
    'Tiempo de establecimiento', 'Topografia', 'Drenaje', 'Riego', 'Fertilizantes aplicados'
]

# Detectar colisiones por mayúsculas, tildes o espacios en cada columna categórica
for col in cat_cols_raw:
    uniq_vals = df_raw[col].dropna().unique()
    tmp = pd.DataFrame({'raw': uniq_vals})
    tmp['key'] = tmp['raw'].map(norm_key)
    coll = tmp.groupby('key')['raw'].apply(list)
    coll = coll[coll.str.len() > 1]
    print(f'[{col}] categorías crudas={len(uniq_vals)} | grupos con variantes (tilde/caso/espacio)={len(coll)}')
    for k, vals in list(coll.items())[:10]:
        counts = {v: int((df_raw[col] == v).sum()) for v in vals}
        print(f'   {k!r} -> {counts}')

[Departamento] categorías crudas=33 | grupos con variantes (tilde/caso/espacio)=0
[Municipio] categorías crudas=990 | grupos con variantes (tilde/caso/espacio)=9
   'ancuya' -> {'ANCUYA': 25, 'ANCUYÁ': 6}
   'bochalema' -> {'BOCHALEMA': 14, 'bOCHALEMA': 1}
   'chima' -> {'CHIMA': 12, 'CHIMÁ': 9}
   'coloso' -> {'COLOSÓ': 1, 'COLOSO': 71}
   'cuaspud' -> {'CUASPUD': 3, 'CUASPÚD': 42}
   'guapi' -> {'GUAPI': 87, 'GUAPÍ': 10}
   'pandi' -> {'Pandi': 9, 'PANDI': 48}
   'potosi' -> {'POTOSÍ': 146, 'POTOSí': 100}
   'sotara' -> {'SOTARÁ': 9, 'SOTARA': 188}
[Cultivo] categorías crudas=282 | grupos con variantes (tilde/caso/espacio)=4


   'boton de oro' -> {'Boton de Oro': 3, 'Botón de Oro': 3}
   'tomate de arbol' -> {'Tomate de Arbol': 82, 'Tomate de Árbol': 79}


   'trebol' -> {'Trebol': 3, 'Trébol': 16}
   'uva caimarona' -> {'Uva caimarona': 1, 'Uva Caimarona': 3}
[Estado] categorías crudas=4 | grupos con variantes (tilde/caso/espacio)=1
   'establecido' -> {'Establecido': 25863, 'ESTABLECIDO': 18244}


[Tiempo de establecimiento] categorías crudas=9 | grupos con variantes (tilde/caso/espacio)=3
   'de 1 a 5 anos' -> {'De 1 a 5 años': 7302, 'de 1 a 5 años': 5752}
   'de 5 a 10 anos' -> {'De 5 a 10 años': 4054, 'de 5 a 10 años': 3348}
   'mas de 10 anos' -> {'Mas de 10 años': 3597, 'mas de 10 años': 2175}
[Topografia] categorías crudas=13 | grupos con variantes (tilde/caso/espacio)=0
[Drenaje] categorías crudas=6 | grupos con variantes (tilde/caso/espacio)=0
[Riego] categorías crudas=17 | grupos con variantes (tilde/caso/espacio)=1
   'no indica' -> {'No indica': 22683, 'No Indica': 16485}


[Fertilizantes aplicados] categorías crudas=5561 | grupos con variantes (tilde/caso/espacio)=193


   '10-20-30, micronfos' -> {'10-20-30, Micronfos': 1, '10-20-30, micronfos': 1}
   '10-30-10 15-15-15' -> {'10-30-10 15-15-15': 2, '10-30-10    15-15-15': 1}
   '10-30-10 y dap' -> {'10-30-10 Y DAP': 1, '10-30-10 y DAP': 1}


   '10-30-10 y urea' -> {'10-30-10 Y ÚREA': 1, '10-30-10 Y UREA': 1}
   '10-30-10, kcl, urea' -> {'10-30-10, KCL, urea': 3, '10-30-10, KCL, Urea': 1}
   '13266 151515' -> {'13266 151515': 14, '13266  151515': 1}
   '15 15 15' -> {'15 15 15': 21, '15  15  15': 1}
   '15-15-15 13-26-6' -> {'15-15-15 13-26-6': 1, '15-15-15    13-26-6': 31, '15-15-15     13-26-6': 5}
   '15-15-15 18-18-18' -> {'15-15-15  18-18-18': 1, '15-15-15     18-18-18': 1, '15-15-15    18-18-18': 1, '15-15-15 18-18-18': 1}
   '15-15-15 agrimins' -> {'15-15-15 AGRIMINS': 17, '15-15-15 Agrimins': 1}


In [11]:
# Verificación geográfica de Municipio condicionada por Departamento (evitar colisión CHIMA vs CHIMÁ)
muni_dep = df_raw[['Departamento', 'Municipio']].dropna().drop_duplicates().copy()
muni_dep['key'] = muni_dep['Municipio'].map(norm_key)
muni_coll_dep = muni_dep.groupby(['Departamento', 'key'])['Municipio'].apply(list)
muni_coll_dep = muni_coll_dep[muni_coll_dep.str.len() > 1]

print('Colisiones de Municipio DENTRO del mismo Departamento:')
for (dep, k), vals in muni_coll_dep.items():
    counts = {v: int(((df_raw['Departamento'] == dep) & (df_raw['Municipio'] == v)).sum()) for v in vals}
    print(f'  {dep} | {k!r} -> {counts}')

print('\nPrueba de que CHIMA (Santander) y CHIMÁ (Córdoba) son municipios distintos:')
display(df_raw[df_raw['Municipio'].fillna('').str.upper().str.contains('CHIM')].groupby(['Departamento', 'Municipio']).size().to_frame('conteo'))

Colisiones de Municipio DENTRO del mismo Departamento:
  CAUCA | 'guapi' -> {'GUAPI': 87, 'GUAPÍ': 10}
  CAUCA | 'sotara' -> {'SOTARÁ': 9, 'SOTARA': 188}
  CUNDINAMARCA | 'pandi' -> {'Pandi': 9, 'PANDI': 48}
  NARIÑO | 'ancuya' -> {'ANCUYA': 25, 'ANCUYÁ': 6}
  NARIÑO | 'cuaspud' -> {'CUASPUD': 3, 'CUASPÚD': 42}


  NARIÑO | 'potosi' -> {'POTOSÍ': 146, 'POTOSí': 100}
  NORTE DE SANTANDER | 'bochalema' -> {'BOCHALEMA': 14, 'bOCHALEMA': 1}


  SUCRE | 'coloso' -> {'COLOSÓ': 1, 'COLOSO': 71}

Prueba de que CHIMA (Santander) y CHIMÁ (Córdoba) son municipios distintos:


,,conteo
Departamento,Municipio,
CESAR,CHIMICHAGUA,58
CÓRDOBA,CHIMÁ,9
SANTANDER,CHIMA,12


In [12]:
# Inspección detallada de variantes en Cultivo (singular/plural y ortografía) y otras categóricas de baja cardinalidad
vc_cult = df_raw['Cultivo'].value_counts()
print('Cultivo — pares singular/plural y ortográficos relevantes:')
for par in [('Pasto', 'Pastos'), ('Estevia', 'Stevia'), ('Boton de Oro', 'Botón de Oro'),
            ('Tomate de Arbol', 'Tomate de Árbol'), ('Trebol', 'Trébol'), ('Uva caimarona', 'Uva Caimarona'),
            ('Bananito', 'Banano'), ('Maní forrajero', 'Maíz Forrajero')]:
    print(f'  {par[0]}: {vc_cult.get(par[0], 0):,}  |  {par[1]}: {vc_cult.get(par[1], 0):,}')

print('\nFrecuencias crudas de Estado, Tiempo de establecimiento, Topografia, Drenaje y Riego:')
for col in ['Estado', 'Tiempo de establecimiento', 'Topografia', 'Drenaje', 'Riego']:
    print(f'--- {col} ---')
    display(df_raw[col].value_counts(dropna=False).to_frame('conteo'))

Cultivo — pares singular/plural y ortográficos relevantes:
  Pasto: 5,886  |  Pastos: 7,542
  Estevia: 1  |  Stevia: 4
  Boton de Oro: 3  |  Botón de Oro: 3
  Tomate de Arbol: 82  |  Tomate de Árbol: 79
  Trebol: 3  |  Trébol: 16
  Uva caimarona: 1  |  Uva Caimarona: 3
  Bananito: 76  |  Banano: 381
  Maní forrajero: 1  |  Maíz Forrajero: 21

Frecuencias crudas de Estado, Tiempo de establecimiento, Topografia, Drenaje y Riego:
--- Estado ---


,conteo
Estado,
Por establecer,33138
Establecido,25863
ESTABLECIDO,18244
No indica,15493


--- Tiempo de establecimiento ---


,conteo
Tiempo de establecimiento,
De 0 a 1 año,35644
No indica,30814
De 1 a 5 años,7302
de 1 a 5 años,5752
De 5 a 10 años,4054
Mas de 10 años,3597
de 5 a 10 años,3348
mas de 10 años,2175
No aplica,52


--- Topografia ---


,conteo
Topografia,
Plano,30656
No indica,12555
Pendiente moderada,10744
Ondulado,10496
Moderadamente ondulado,9839
Pendiente,9245
Ligeramente ondulado,2997
Pendiente leve,1942
Ondulado y Pendiente,1385


--- Drenaje ---


,conteo
Drenaje,
Buen drenaje,49807
No indica,25605
Regular drenaje,14229
Mal drenaje,2667
Muy buen drenaje,404
Muy mal drenaje,26


--- Riego ---


,conteo
Riego,
No Tiene,41159
No indica,22683
No Indica,16485
Aspersión,4836
Gravedad,3133
Goteo,2820
Manguera,682
Por Inundación,463
Aspersión - Goteo,105


**Hallazgos 2.5:**
* **`Departamento`:** 33 valores oficiales limpios (`32` departamentos + `BOGOTÁ, D.C.`), sin variantes.
* **`Municipio`:** `2` valores `NaN` en `NARIÑO` y **8 pares con variantes tipográficas dentro del mismo departamento** (`GUAPI`/`GUAPÍ`, `SOTARA`/`SOTARÁ` en `CAUCA`; `PANDI`/`Pandi` en `CUNDINAMARCA`; `ANCUYA`/`ANCUYÁ`, `CUASPUD`/`CUASPÚD`, `POTOSÍ`/`POTOSí` en `NARIÑO`; `BOCHALEMA`/`bOCHALEMA` en `NORTE DE SANTANDER`; `COLOSO`/`COLOSÓ` en `SUCRE`). En cambio, `CHIMA` (Santander) y `CHIMÁ` (Córdoba) son municipios homónimos en departamentos distintos y **no** deben fusionarse.
* **`Cultivo`:** `'Pasto'` (`5,886`) y `'Pastos'` (`7,542`) representan la misma cobertura forrajera; al unificarse suman `13,428` y pasan a ser la primera categoría del país (por encima de `'Cacao'` con `11,104`). Además aparecen 4 colisiones por tilde/mayúscula (`Botón de Oro`, `Tomate de Árbol`, `Trébol`, `Uva Caimarona`) y `'Estevia'`/`'Stevia'`. Se respeta la distinción biológica entre `'Bananito'`/`'Banano'` y `'Maní forrajero'`/`'Maíz Forrajero'`.
* **`Estado`, `Tiempo de establecimiento`, `Topografia`, `Riego`, `Fertilizantes aplicados`:** presentan duplicaciones por mayúsculas (`'Establecido'`/`'ESTABLECIDO'`, `'de 1 a 5 años'`/`'De 1 a 5 años'`, `'No indica'`/`'No Indica'`, `'SI'`/`'Si'`/`'si'`), falta de tilde (`'Mas de 10 años'`, `'Cañon'`), falta de espacios alrededor de guion (`'Superficial-Aspersión'`) y mezcla entre `'No indica'` (dato faltante) y categorías válidas como `'No Tiene'` (sin riego) o `'No aplica'` (tiempo de establecimiento).

---
### 2.6 Diagnóstico de valores imposibles fisicoquímicamente e inconsistencias

In [13]:
# Conversión temporal auxiliar solo para auditar rangos y valores imposibles en bruto
def parse_temp(series, comma_decimal=False):
    x = series.dropna().astype(str).str.strip()
    x = x[x.str.upper() != 'ND']
    if comma_decimal:
        x = x.str.replace(',', '.', regex=False)
    x = x.str.replace('4..01', '4.01', regex=False)
    x = x.str.replace(r'^[<>]\s*', '', regex=True)
    return pd.to_numeric(x, errors='coerce')

range_audit = []
for col in chem_cols_raw:
    is_cdec = ('doble' in col.lower())
    num = parse_temp(df_raw[col], comma_decimal=is_cdec)
    range_audit.append({
        'columna': col,
        'n_validos': int(num.notna().sum()),
        'minimo': round(float(num.min()), 4),
        'mediana': round(float(num.median()), 4),
        'p99': round(float(num.quantile(0.99)), 4),
        'maximo': round(float(num.max()), 4),
        'negativos_lt_0': int((num < 0).sum()),
        'ceros_exactos': int((num == 0).sum()),
    })

display(pd.DataFrame(range_audit))

print('Casos específicos imposibles o malformados detectados:')
display(df_raw[
    (df_raw['pH agua:suelo'].astype(str).str.strip() == '4..01') |
    (df_raw['Azufre Fosfato monocalcico'].astype(str).str.strip().str.startswith('-')) |
    (df_raw['Sodio intercambiable'].astype(str).str.strip().str.startswith('-')) |
    (df_raw['Boro disponible'].astype(str).str.strip().isin(['b', '-0.00']))
][['Secuencial', 'Fecha de Análisis', 'Departamento', 'Cultivo', 'pH agua:suelo',
   'Acidez Intercambiable', 'Azufre Fosfato monocalcico', 'Sodio intercambiable', 'Boro disponible']])

# Verificar pH fuera de [0, 14] y Materia Orgánica > 100%
ph_num = parse_temp(df_raw['pH agua:suelo'])
mo_num = parse_temp(df_raw['Materia organica'])
print(f'pH < 0 o > 14: {int(((ph_num < 0) | (ph_num > 14)).sum())} | Rango observado: [{ph_num.min()}, {ph_num.max()}]')
print(f'Materia orgánica > 100%: {int((mo_num > 100).sum())} | Registros > 50%: {int((mo_num > 50).sum())} (máx = {mo_num.max()}%)')

# Verificar inconsistencia analítica menor: Aluminio intercambiable > Acidez Intercambiable
ac_num = parse_temp(df_raw['Acidez Intercambiable'])
al_num = parse_temp(df_raw['Aluminio intercambiable'])
al_gt_ac = (al_num > ac_num)
print(f'Registros con Aluminio > Acidez intercambiable: {int(al_gt_ac.sum())} (diferencia máxima: {(al_num - ac_num).max():.3f} cmol(+)/kg)')

,columna,n_validos,minimo,mediana,p99,maximo,negativos_lt_0,ceros_exactos
0,pH agua:suelo,92738,3.190,5.470,8.4300,10.57,0,0
1,Materia organica,92736,0.000,2.750,21.8200,75.04,0,1
2,Fósforo Bray II,91269,0.088,8.403,315.1064,3015.00,0,0
3,Azufre Fosfato monocalcico,91239,-1.010,6.202,166.2868,4862.00,1,0
4,Acidez Intercambiable,47608,0.000,1.692,11.0886,40.89,0,602
5,Aluminio intercambiable,47608,0.000,1.315,9.3393,36.60,0,2917
6,Calcio intercambiable,92738,0.028,4.130,45.9126,171.80,0,0
7,Magnesio intercambiable,92738,0.006,1.020,12.8500,68.52,0,0
8,Potasio intercambiable,92738,0.090,0.220,2.2800,64.06,0,0
9,Sodio intercambiable,92738,-0.040,0.140,2.2000,99.11,1,6


Casos específicos imposibles o malformados detectados:


,Secuencial,Fecha de Análisis,Departamento,Cultivo,pH agua:suelo,Acidez Intercambiable,Azufre Fosfato monocalcico,Sodio intercambiable,Boro disponible
4505,"4,505",10/10/2014,ANTIOQUIA,Uchuva,5.64,0.078,16.77,0.087,b
7282,"7,283",8/01/2014,CASANARE,Arveja,4..01,0.787,1.978,0.027,0.167
13435,"13,436",2/10/2015,NORTE DE SANTANDER,Pastos,5.15,0.892,3.126,0.163,-0.00
14703,"14,703",4/12/2015,BOYACÁ,Caña Panelera,6.28,ND,-1.01,0.230,0.487
15763,"15,764",21/12/2015,SANTANDER,Tabaco,8.210,ND,18.83,-0.04,0.631
48092,"48,078",27/11/2018,VALLE DEL CAUCA,Hortalizas Varias,5.98,ND,5.685,< 0.1,-0.00


pH < 0 o > 14: 0 | Rango observado: [3.19, 10.57]
Materia orgánica > 100%: 0 | Registros > 50%: 17 (máx = 75.04%)
Registros con Aluminio > Acidez intercambiable: 36 (diferencia máxima: 4.227 cmol(+)/kg)


**Hallazgos 2.6:**
* **Escala de pH (`[0, 14]`):** `0` valores fuera del rango físico (`3.19` a `10.57`), pero existe **1 registro malformado (`'4..01'`, Secuencial `7,283`)** cuya `Acidez Intercambiable = 0.787 cmol(+)/kg` confirma que pertenece a un suelo ácido (`pH ≈ 4.01`).
* **Concentraciones negativas (físicamente imposibles):** exactamente **2 valores negativos estrictos**: `Azufre Fosfato monocalcico = -1.01` (Secuencial `14,703`) y `Sodio intercambiable = -0.04` (Secuencial `15,764`). Además hay **2 ceros negativos (`-0.00`)** y **1 carácter no numérico (`'b'`, Secuencial `4,505`)** en `Boro disponible`.
* **Materia orgánica (`> 100%`):** `0` casos imposibles (máximo `75.04%`); los `17` registros con `MO > 50%` corresponden a suelos orgánicos/histosoles plausibles en páramos o zonas húmedas de Boyacá, Putumayo, Valle del Cauca y Cundinamarca, por lo que se **conservan**.
* **Aluminio > Acidez intercambiable:** `36` registros muestran `Al > Acidez` (máximo `4.227 cmol(+)/kg` por encima), atribuible a incertidumbre diferencial entre dos determinaciones titulométricas/espectrofotométricas; se **conservan sin alterar** y se dejan documentados.

---
## 3. Decisiones de limpieza y transformación en `pandas`

Cada problema diagnosticado recibe una regla explícita y justificada:

| Problema detectado | Regla aplicada | Justificación |
| :--- | :--- | :--- |
| Encabezados con `\xa0`, tildes, `:` y mayúsculas mixtas | Normalizar a `snake_case` sin tildes y con sufijo de unidades | Evita errores ocultos (`\xa0`) y preserva trazabilidad de unidades (`pct`, `mg_kg`, `cmol_kg`, `ds_m`) |
| `2,046` filas idénticas al ignorar `Secuencial` | **Eliminar** duplicados conservando la primera ocurrencia (`90,692` filas) | Evita sobreponderar análisis repetidos en estadísticos y modelos |
| `Secuencial` con comas de miles | Quitar `,` y convertir a `Int64` | Recupera el identificador numérico entero |
| `Fecha de Análisis` como texto | Convertir con `format='%d/%m/%Y'` a `datetime64` | Garantiza orden cronológico exacto sin invertir día y mes (`0 NaT`) |
| `'ND'` y `'No indica'` / `'No Indica'` | **Sustituir por `NaN`** (conservando `'No Tiene'` y `'No aplica'`) | Diferencia ausencia de dato de categorías negativas reales (sin riego / no aplica) |
| Coma decimal en `Hierro` (`42`) y `Zinc` (`4,732`) doble ácido | Reemplazar `,` por `.` antes del parseo numérico | Corrige el formato decimal mixto sin distorsionar órdenes de magnitud |
| Censurados `<LD` en 20 analitos de concentración/CE | **Imputar a `LD / 2`** (`< 0.0` → `0.0`) | Criterio estándar ambiental/geoquímico (Helsel/EPA) para datos censurados a la izquierda en escala de razón |
| Censurados en pH (`<3.80`, `>10.00`) y cotas `>UL` (`MO`, `CE`) | **Conservar el umbral numérico** (`3.80`, `10.00`, `44.82`, `12.88`) | El pH es logarítmico (`3.80 / 2 = 1.90` sería físicamente falso); para `>UL` es la cota inferior observable |
| `'4..01'` en pH | **Corregir** tipografía a `4.01` | Coherente con `Acidez Intercambiable = 0.787 cmol(+)/kg` del mismo registro |
| `'b'` en Boro y negativos (`-1.01` Azufre, `-0.04` Sodio) | **Sustituir por `NaN`** (`-0.00` → `0.0`) | Concentraciones negativas son imposibles y `'b'` es error de digitación sin valor recuperable |
| `2` municipios `NaN` y `36` filas `Al > Acidez` | **Conservar** tal cual (`NaN` / valores originales) | No es válido inventar un municipio ni sobrescribir mediciones reales de laboratorio |
| Variantes tipográficas en `Municipio` | Normalizar mayúsculas y mapear solo colisiones dentro del mismo `Departamento` | Corrige tildes inconsistentes sin colapsar `CHIMA` (Santander) con `CHIMÁ` (Córdoba) |
| Variantes en `Cultivo`, `Estado`, `Tiempo`, `Topografía`, `Riego`, `Fertilizantes` | Estandarizar ortografía, tildes y singular/plural (`Pasto` → `Pastos`) | Elimina fragmentación artificial de categorías idénticas |

In [14]:
# Copia de trabajo limpia y mapeo exacto de encabezados a snake_case sin tildes ni \xa0
df = df_raw.copy()
df.columns = [re.sub(r'\s+', ' ', c).strip() for c in df.columns]

RENAME_MAP = {
    'Secuencial': 'secuencial',
    'Fecha de Análisis': 'fecha_analisis',
    'Departamento': 'departamento',
    'Municipio': 'municipio',
    'Cultivo': 'cultivo',
    'Estado': 'estado_cultivo',
    'Tiempo de establecimiento': 'tiempo_establecimiento',
    'Topografia': 'topografia',
    'Drenaje': 'drenaje',
    'Riego': 'riego',
    'Fertilizantes aplicados': 'fertilizantes_aplicados',
    'pH agua:suelo': 'ph_agua_suelo',
    'Materia organica': 'materia_organica_pct',
    'Fósforo Bray II': 'fosforo_bray_ii_mg_kg',
    'Azufre Fosfato monocalcico': 'azufre_mg_kg',
    'Acidez Intercambiable': 'acidez_intercambiable_cmol_kg',
    'Aluminio intercambiable': 'aluminio_intercambiable_cmol_kg',
    'Calcio intercambiable': 'calcio_intercambiable_cmol_kg',
    'Magnesio intercambiable': 'magnesio_intercambiable_cmol_kg',
    'Potasio intercambiable': 'potasio_intercambiable_cmol_kg',
    'Sodio intercambiable': 'sodio_intercambiable_cmol_kg',
    'capacidad de intercambio cationico': 'cic_cmol_kg',
    'Conductividad electrica': 'conductividad_electrica_ds_m',
    'Hierro disponible olsen': 'hierro_olsen_mg_kg',
    'Cobre disponible': 'cobre_olsen_mg_kg',
    'Manganeso disponible Olsen': 'manganeso_olsen_mg_kg',
    'Zinc disponible Olsen': 'zinc_olsen_mg_kg',
    'Boro disponible': 'boro_mg_kg',
    'Hierro disponible doble acido': 'hierro_doble_acido_mg_kg',
    'Cobre disponible doble acido': 'cobre_doble_acido_mg_kg',
    'Manganeso disponible doble acido': 'manganeso_doble_acido_mg_kg',
    'Zinc disponible doble acido': 'zinc_doble_acido_mg_kg',
}
assert set(df.columns) == set(RENAME_MAP.keys()), 'Falta mapear alguna columna'
df = df.rename(columns=RENAME_MAP)

# 1. Eliminar filas duplicadas ignorando 'secuencial'
cols_dedup = [c for c in df.columns if c != 'secuencial']
n_antes = len(df)
df = df.drop_duplicates(subset=cols_dedup, keep='first').reset_index(drop=True)
print(f'Eliminación de duplicados: {n_antes:,} -> {len(df):,} filas (-{n_antes - len(df):,})')

# 2. Conversión de secuencial y fecha_analisis
df['secuencial'] = pd.to_numeric(
    df['secuencial'].astype(str).str.replace(',', '', regex=False).str.strip(),
    errors='raise'
).astype('Int64')
df['fecha_analisis'] = pd.to_datetime(
    df['fecha_analisis'].astype(str).str.strip(),
    format='%d/%m/%Y',
    errors='raise'
)
print(f'Fechas convertidas: 0 NaT | Rango: {df["fecha_analisis"].min().date()} a {df["fecha_analisis"].max().date()}')

Eliminación de duplicados: 92,738 -> 90,692 filas (-2,046)
Fechas convertidas: 0 NaT | Rango: 2014-01-07 a 2025-09-05


In [15]:
# 3. Conteo de analitos censurados por fila y limpieza/conversión de las 21 columnas fisicoquímicas
chem_cols = [
    'ph_agua_suelo', 'materia_organica_pct', 'fosforo_bray_ii_mg_kg', 'azufre_mg_kg',
    'acidez_intercambiable_cmol_kg', 'aluminio_intercambiable_cmol_kg',
    'calcio_intercambiable_cmol_kg', 'magnesio_intercambiable_cmol_kg',
    'potasio_intercambiable_cmol_kg', 'sodio_intercambiable_cmol_kg',
    'cic_cmol_kg', 'conductividad_electrica_ds_m',
    'hierro_olsen_mg_kg', 'cobre_olsen_mg_kg', 'manganeso_olsen_mg_kg', 'zinc_olsen_mg_kg',
    'boro_mg_kg', 'hierro_doble_acido_mg_kg', 'cobre_doble_acido_mg_kg',
    'manganeso_doble_acido_mg_kg', 'zinc_doble_acido_mg_kg',
]

# Contar cuántos analitos por fila traían censura (< o >) en el dato original
censored_matrix = pd.DataFrame({
    col: df[col].fillna('').astype(str).str.strip().str.match(r'^[<>]')
    for col in chem_cols
})
df['n_analitos_censurados'] = censored_matrix.sum(axis=1).astype('Int64')

def clean_chem_series(series: pd.Series, col_name: str) -> pd.Series:
    s = series.astype('object').where(series.notna(), np.nan)
    s = s.map(lambda v: np.nan if pd.isna(v) else re.sub(r'\s+', '', str(v).strip()))

    if col_name in ('hierro_doble_acido_mg_kg', 'zinc_doble_acido_mg_kg'):
        s = s.map(lambda v: v.replace(',', '.') if isinstance(v, str) else v)

    if col_name == 'ph_agua_suelo':
        s = s.replace('4..01', '4.01')

    # Reemplazar centinelas de texto ('ND' y 'b') por NaN
    s = s.map(lambda v: np.nan if isinstance(v, str) and v.upper() in ('ND', 'B', '') else v)

    # Identificar censurados <LD y >UL antes de retirar el símbolo
    is_lt = s.map(lambda v: isinstance(v, str) and v.startswith('<'))
    s_num_str = s.map(lambda v: v[1:] if isinstance(v, str) and v[0] in ('<', '>') else v)

    num = pd.to_numeric(s_num_str, errors='raise').astype('float64')

    # Para variables de escala de razón (todas excepto pH), sustituir <LD por LD / 2
    if col_name != 'ph_agua_suelo':
        num = num.where(~is_lt, num / 2.0)

    # Valores negativos imposibles (< 0) -> NaN; normalizar -0.0 a +0.0
    num = num.where(num >= 0.0, np.nan)
    num = num.where(num != 0.0, 0.0)
    return num

for col in chem_cols:
    df[col] = clean_chem_series(df[col], col)

print('Conversión de 21 columnas fisicoquímicas completada.')
print('Valores negativos restantes en todas las químicas:', int((df[chem_cols] < 0).sum().sum()))
print('Distribución de n_analitos_censurados:')
display(df['n_analitos_censurados'].value_counts().sort_index().to_frame('registros'))

Conversión de 21 columnas fisicoquímicas completada.
Valores negativos restantes en todas las químicas: 0
Distribución de n_analitos_censurados:


,registros
n_analitos_censurados,
0,27338
1,25400
2,16162
3,10182
4,5444
5,2874
6,1577
7,1016
8,694


In [16]:
# 4. Estandarización de variables categóricas
cat_cols = [
    'departamento', 'municipio', 'cultivo', 'estado_cultivo',
    'tiempo_establecimiento', 'topografia', 'drenaje', 'riego', 'fertilizantes_aplicados'
]

for col in cat_cols:
    df[col] = df[col].map(
        lambda v: np.nan if pd.isna(v) else re.sub(r'\s+', ' ', str(v).strip())
    )

# Departamento: ya limpio en mayúsculas (33 valores)
df['departamento'] = df['departamento'].str.upper()

# Municipio: pasar a mayúsculas limpias y resolver solo las 5 colisiones de tilde dentro del mismo departamento
df['municipio'] = df['municipio'].map(lambda v: v.upper() if isinstance(v, str) else np.nan)
MUNI_CORRECCIONES_DEP = {
    ('CAUCA', 'GUAPÍ'): 'GUAPI',
    ('CAUCA', 'SOTARA'): 'SOTARÁ',
    ('NARIÑO', 'ANCUYA'): 'ANCUYÁ',
    ('NARIÑO', 'CUASPÚD'): 'CUASPUD',
    ('SUCRE', 'COLOSO'): 'COLOSÓ',
}
for (dep, m_old), m_new in MUNI_CORRECCIONES_DEP.items():
    mask = (df['departamento'] == dep) & (df['municipio'] == m_old)
    df.loc[mask, 'municipio'] = m_new

# Cultivo: unificar variantes tipográficas, singular/plural (Pasto -> Pastos) y convertir 'No Indica' en NaN
CULTIVO_MAP = {
    'No Indica': np.nan,
    'Pasto': 'Pastos',
    'Boton de Oro': 'Botón de Oro',
    'Tomate de Arbol': 'Tomate de Árbol',
    'Trebol': 'Trébol',
    'Uva caimarona': 'Uva Caimarona',
    'Estevia': 'Stevia',
}
df['cultivo'] = df['cultivo'].replace(CULTIVO_MAP)

# Estado del cultivo
df['estado_cultivo'] = df['estado_cultivo'].replace({
    'ESTABLECIDO': 'Establecido',
    'No indica': np.nan,
})

# Tiempo de establecimiento (conservar 'No aplica' como categoría válida)
df['tiempo_establecimiento'] = df['tiempo_establecimiento'].replace({
    'de 1 a 5 años': 'De 1 a 5 años',
    'de 5 a 10 años': 'De 5 a 10 años',
    'Mas de 10 años': 'Más de 10 años',
    'mas de 10 años': 'Más de 10 años',
    'No indica': np.nan,
})

# Topografía y Drenaje
df['topografia'] = df['topografia'].replace({
    'Ondulado y Pendiente': 'Ondulado y pendiente',
    'No indica': np.nan,
})
df['drenaje'] = df['drenaje'].replace({'No indica': np.nan})

# Riego (conservar 'No tiene' como categoría válida distinta de NaN)
df['riego'] = df['riego'].replace({
    'No indica': np.nan,
    'No Indica': np.nan,
    'No Tiene': 'No tiene',
    'Superficial-Aspersión': 'Superficial - Aspersión',
    'Cañon': 'Cañón',
    'Por Inundación': 'Por inundación',
})

# Fertilizantes aplicados: unificar variantes de No y Sí, y 'No indica' -> NaN
df['fertilizantes_aplicados'] = df['fertilizantes_aplicados'].replace({
    'No indica': np.nan,
    'NO': 'No',
    'no': 'No',
    'Ninguno': 'No',
    'SI': 'Sí',
    'Si': 'Sí',
    'si': 'Sí',
})

print('Verificación de cardinalidad tras estandarización:')
for col in cat_cols:
    print(f'  {col:25s}: {df[col].nunique(dropna=True):4d} categorías únicas | NaN = {int(df[col].isna().sum()):,}')

Verificación de cardinalidad tras estandarización:
  departamento             :   33 categorías únicas | NaN = 0
  municipio                :  982 categorías únicas | NaN = 2
  cultivo                  :  275 categorías únicas | NaN = 7,469
  estado_cultivo           :    2 categorías únicas | NaN = 15,332
  tiempo_establecimiento   :    5 categorías únicas | NaN = 29,572
  topografia               :   12 categorías únicas | NaN = 12,408
  drenaje                  :    5 categorías únicas | NaN = 25,378
  riego                    :   15 categorías únicas | NaN = 38,401
  fertilizantes_aplicados  : 5524 categorías únicas | NaN = 49,255


In [17]:
# 5. Creación de variables derivadas (6 columnas nuevas -> 38 columnas totales)
df['anio_analisis'] = df['fecha_analisis'].dt.year.astype('Int64')
df['mes_analisis'] = df['fecha_analisis'].dt.month.astype('Int64')

# Clasificación de acidez/alcalinidad del suelo según pH (escala USDA / IGAC)
ph_bins = [-np.inf, 4.5, 5.5, 6.0, 6.6, 7.4, 7.9, 8.5, np.inf]
ph_labels = [
    'Extremadamente ácido (<4.5)',
    'Muy a fuertemente ácido (4.5-5.5)',
    'Moderadamente ácido (5.5-6.0)',
    'Ligeramente ácido (6.0-6.6)',
    'Neutro (6.6-7.4)',
    'Ligeramente alcalino (7.4-7.9)',
    'Moderadamente alcalino (7.9-8.5)',
    'Fuertemente alcalino (>=8.5)',
]
df['clase_ph'] = pd.cut(
    df['ph_agua_suelo'], bins=ph_bins, labels=ph_labels, right=False
).astype('object')

# Indicador del método de extracción de micronutrientes
df['metodo_micronutrientes'] = np.select(
    [df['hierro_olsen_mg_kg'].notna(), df['hierro_doble_acido_mg_kg'].notna()],
    ['Olsen', 'Doble ácido'],
    default='Sin determinar',
)

# Suma de bases intercambiables (Ca + Mg + K + Na) en cmol(+)/kg
df['suma_bases_cmol_kg'] = (
    df['calcio_intercambiable_cmol_kg']
    + df['magnesio_intercambiable_cmol_kg']
    + df['potasio_intercambiable_cmol_kg']
    + df['sodio_intercambiable_cmol_kg']
)

print('Resumen de variables derivadas creadas:')
display(df['clase_ph'].value_counts(dropna=False).to_frame('registros'))
display(df['metodo_micronutrientes'].value_counts(dropna=False).to_frame('registros'))
display(df[['anio_analisis', 'mes_analisis', 'n_analitos_censurados', 'suma_bases_cmol_kg']].describe().round(3))

Resumen de variables derivadas creadas:


,registros
clase_ph,
Muy a fuertemente ácido (4.5-5.5),40670
Moderadamente ácido (5.5-6.0),15716
Ligeramente ácido (6.0-6.6),11767
Neutro (6.6-7.4),8973
Extremadamente ácido (<4.5),5725
Moderadamente alcalino (7.9-8.5),3938
Ligeramente alcalino (7.4-7.9),3238
Fuertemente alcalino (>=8.5),665


,registros
metodo_micronutrientes,
Olsen,84873
Doble ácido,4733
Sin determinar,1086


,anio_analisis,mes_analisis,n_analitos_censurados,suma_bases_cmol_kg
count,90692.0,90692.0,90692.0,90691.000
mean,2018.439,7.256,1.616,10.002
std,3.387,3.399,1.677,11.493
min,2014.0,1.0,0.0,0.165
25%,2015.0,5.0,0.0,2.182
50%,2018.0,8.0,1.0,5.910
75%,2022.0,10.0,2.0,13.552
max,2025.0,12.0,9.0,206.780


---
## 4. Reporte comparativo Antes vs. Después, Bitácora técnica y Exportación

### 4.1 Tabla comparativa Antes vs. Después de la limpieza

In [18]:
# Conteo de faltantes efectivos y columnas por tipo antes vs. después
def count_effective_missing_raw(frame):
    total = 0
    for c in frame.columns:
        s = frame[c]
        total += int(s.isna().sum())
        sc = s.dropna().astype(str).str.strip()
        total += int(sc.str.upper().eq('ND').sum())
        total += int(sc.str.lower().eq('no indica').sum())
    return total

num_cols_raw = sum(pd.api.types.is_numeric_dtype(df_raw[c]) for c in df_raw.columns)
dt_cols_raw = sum(pd.api.types.is_datetime64_any_dtype(df_raw[c]) for c in df_raw.columns)
num_cols_clean = sum(pd.api.types.is_numeric_dtype(df[c]) for c in df.columns)
dt_cols_clean = sum(pd.api.types.is_datetime64_any_dtype(df[c]) for c in df.columns)

ram_raw_mb = df_raw.memory_usage(deep=True).sum() / (1024**2)
ram_clean_mb = df.memory_usage(deep=True).sum() / (1024**2)

comparativa = pd.DataFrame([
    ('Filas totales', f'{len(df_raw):,}', f'{len(df):,}', f'-{len(df_raw) - len(df):,} (-2.21%)'),
    ('Columnas totales', f'{df_raw.shape[1]}', f'{df.shape[1]}', '+6 derivadas'),
    ('Columnas numéricas tipadas', f'{num_cols_raw}', f'{num_cols_clean}', f'+{num_cols_clean}'),
    ('Columnas de fecha tipadas', f'{dt_cols_raw}', f'{dt_cols_clean}', '+1'),
    ('Duplicados (sin contar ID)', '2,046', '0', '-2,046'),
    ('Valores NaN explícitos', f'{int(df_raw.isna().sum().sum()):,}', f'{int(df.isna().sum().sum()):,}', 'Transparentados desde ND / No indica'),
    ('Faltantes reales totales (NaN + centinelas)', f'{count_effective_missing_raw(df_raw):,}', f'{int(df.isna().sum().sum()):,}', 'Unificados bajo NaN estándar'),
    ('Valores corruptos / negativos imposibles', '5 (+2 ceros -0.00)', '0', 'Corregidos o pasados a NaN'),
    ('Categorías únicas en Cultivo', f'{df_raw["Cultivo"].nunique()}', f'{df["cultivo"].nunique()}', f'-{df_raw["Cultivo"].nunique() - df["cultivo"].nunique()} variantes'),
    ('Categorías únicas en Estado', f'{df_raw["Estado"].nunique()}', f'{df["estado_cultivo"].nunique()}', '-2 (Establecido + No indica->NaN)'),
    ('Categorías únicas en Riego', f'{df_raw["Riego"].nunique()}', f'{df["riego"].nunique()}', '-3 (No indica->NaN + ortografía)'),
    ('Memoria en RAM (MB)', f'{ram_raw_mb:.2f}', f'{ram_clean_mb:.2f}', f'{ram_clean_mb - ram_raw_mb:+.2f} MB'),
], columns=['Métrica', 'Antes (df_raw)', 'Después (df)', 'Cambio / Resultado'])

display(comparativa)

,Métrica,Antes (df_raw),Después (df),Cambio / Resultado
0,Filas totales,"92,738","90,692","-2,046 (-2.21%)"
1,Columnas totales,32,38,+6 derivadas
2,Columnas numéricas tipadas,0,26,+26
3,Columnas de fecha tipadas,0,1,+1
4,Duplicados (sin contar ID),"2,046",0,"-2,046"
5,Valores NaN explícitos,3,"638,440",Transparentados desde ND / No indica
6,Faltantes reales totales (NaN + centinelas),"652,157","638,440",Unificados bajo NaN estándar
7,Valores corruptos / negativos imposibles,5 (+2 ceros -0.00),0,Corregidos o pasados a NaN
8,Categorías únicas en Cultivo,282,275,-7 variantes
9,Categorías únicas en Estado,4,2,-2 (Establecido + No indica->NaN)


### 4.2 Bitácora técnica inicial (problemas reales enfrentados y soluciones aplicadas)

1. **Entorno de Python sin librerías científicas:** El intérprete Python 3.15 instalado por defecto en el sistema no contaba con `pandas`, `matplotlib` ni `seaborn`. Se resolvió creando un entorno virtual local aislado `.venv` con `uv` sobre CPython 3.14.7 e instalando versiones reproducibles (`pandas`, `numpy`, `matplotlib`, `seaborn`, `ipykernel`, `nbformat`, `nbclient`).
2. **Carácter no rompible oculto (`\xa0`) en encabezados:** La columna `'Zinc disponible doble \xa0acido'` contenía un espacio U+00A0 indetectable visualmente. Se resolvió normalizando espacios Unicode con `re.sub(r'\s+', ' ', c).strip()` antes de mapear todos los nombres a `snake_case` sin tildes y con unidades explícitas.
3. **Falsa completitud al usar `.isna()`:** En el archivo crudo solo había `3` celdas `NaN` explícitas (`2` en `Municipio` y `1` en `Hierro disponible doble acido`), mientras que `'ND'` y `'No indica'`/`'No Indica'` ocultaban cientos de miles de datos ausentes y forzaban a las 21 columnas numéricas a cargarse como texto (`str`). Se auditó y convirtió cada centinela a `NaN`, respetando categorías negativas válidas (`'No tiene'` en riego y `'No aplica'` en tiempo de establecimiento).
4. **Duplicados enmascarados por la columna `Secuencial`:** `df_raw.duplicated().sum()` reportaba `0`, pero al excluir el consecutivo se detectaron **2,046 filas redundantes** (`2014–2019`), que fueron eliminadas conservando la primera aparición (`92,738` → `90,692` filas).
5. **Doble significado de la coma `,` dentro del mismo CSV:** `Secuencial` usa `,` como separador de miles (`1,000`–`92,738`), mientras `Zinc disponible doble acido` (`4,732` filas) y `Hierro disponible doble acido` (`42` filas) la usan como separador decimal. Leer con `thousands=','` habría multiplicado por `1,000` concentraciones `< 10 mg/kg`; se resolvió cargando como texto crudo y tratando cada columna según su formato real.
6. **Riesgo de colisión geográfica entre departamentos en `Municipio`:** `CHIMA` (Santander, `12` muestras) y `CHIMÁ` (Córdoba, `9` muestras) colisionan si se remueven tildes globalmente. Se resolvió aplicando la estandarización de tildes condicionada a pares `(departamento, municipio)`.
7. **Censura analítica (`<LD`, `>UL`) y valores corruptos/imposibles:** Se detectaron miles de valores bajo límite de detección (`<LD`), cotas `>UL`, `'4..01'` en pH, `'b'` en Boro y dos valores negativos imposibles (`Azufre = -1.01`, `Sodio = -0.04`). Se aplicó `LD / 2` únicamente en escala de razón, se preservaron las cotas en pH logarítmico (`3.80`, `10.00`) y `>UL`, se corrigió `'4..01'` y se pasaron a `NaN` los valores corruptos o negativos.
8. **Faltantes estructurales por protocolo de laboratorio:** Las tablas cruzadas demostraron que `Acidez/Aluminio` (`48.66%` `'ND'`) corresponden casi en su totalidad (`99.1%`) a suelos con `pH > 5.5`, y que `Olsen` y `Doble ácido` son métodos mutuamente excluyentes (`0` filas con ambos). Esto evitó imputaciones estadísticas erróneas y motivó la creación de la variable derivada `metodo_micronutrientes`.

---
### 4.3 Exportación a `data_processed/suelos_limpios.csv` y verificación integral

In [19]:
os.makedirs('data_processed', exist_ok=True)
df.to_csv(PROCESSED_PATH, index=False, encoding='utf-8')
out_bytes = os.path.getsize(PROCESSED_PATH)
print(f'Archivo exportado: {PROCESSED_PATH} ({out_bytes:,} bytes | {out_bytes / (1024**2):.2f} MB)')

# Relectura y validaciones automáticas post-exportación
df_check = pd.read_csv(PROCESSED_PATH, parse_dates=['fecha_analisis'], low_memory=False)

assert df_check.shape == (90692, 38), f'Dimensiones inesperadas: {df_check.shape}'
assert df_check.duplicated(subset=df_check.columns.drop('secuencial')).sum() == 0, 'Quedaron duplicados'
assert df_check['ph_agua_suelo'].between(0, 14).all(), 'pH fuera de [0, 14]'
for col in chem_cols:
    vals = df_check[col].dropna()
    assert (vals >= 0).all(), f'Valores negativos en {col}'
assert df_check['fecha_analisis'].isna().sum() == 0, 'NaT en fecha_analisis'
assert df_check['clase_ph'].isna().sum() == 0, 'NaN en clase_ph'

print('TODAS las aserciones de integridad post-exportación pasaron con éxito.')
print('\nEstructura final (df_check.info()):')
df_check.info()

Archivo exportado: data_processed/suelos_limpios.csv (22,441,420 bytes | 21.40 MB)


TODAS las aserciones de integridad post-exportación pasaron con éxito.

Estructura final (df_check.info()):
<class 'pandas.DataFrame'>
RangeIndex: 90692 entries, 0 to 90691
Data columns (total 38 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   secuencial                       90692 non-null  int64         
 1   fecha_analisis                   90692 non-null  datetime64[us]
 2   departamento                     90692 non-null  str           
 3   municipio                        90690 non-null  str           
 4   cultivo                          83223 non-null  str           
 5   estado_cultivo                   75360 non-null  str           
 6   tiempo_establecimiento           61120 non-null  str           
 7   topografia                       78284 non-null  str           
 8   drenaje                          65314 non-null  str           
 9   riego                         